# Since  the file contains line-delimited JSON (each line is a JSON object):

In [34]:
import pandas as pd

df=pd.read_json("Dataset_Assignment.JSON",lines=True)


In [35]:
df.shape

(51, 2)

In [36]:
df.head()

,text,labels
0,Floor length chiffon bridesmaid dress with ple...,"{'Silhouette': 'A-line', 'Fabric': 'chiffon', ..."
1,Sparkly sequin fitted prom gown featuring a de...,"{'Silhouette': 'fitted', 'Fabric': 'sequin', '..."
2,Off shoulder satin ball gown with corset bodic...,"{'Silhouette': 'ball gown', 'Fabric': 'satin',..."
3,Lace mermaid wedding dress with long sleeves a...,"{'Silhouette': 'mermaid', 'Fabric': 'lace', 'N..."
4,Short cocktail dress with feather trim and bea...,"{'Silhouette': 'sheath', 'Fabric': 'unknown', ..."


In [37]:
import spacy
from spacy.tokens import DocBin
import json
import os

def create_spacy_data(input_file, output_file):
    nlp = spacy.blank("en")
    doc_bin = DocBin()
    
    count = 0
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            item = json.loads(line)
            text = item['text']
            labels = item['labels']
            
            doc = nlp.make_doc(text)
            ents = []
            
            for label_name, value in labels.items():
                # Split multiple values like "sage; dusty blue"
                sub_values = [v.strip() for v in str(value).split(';')]
                
                for val in sub_values:
                    if val.lower() in ["none", "unknown", "n/a"]:
                        continue
                    
                    # Find the exact position of the attribute in the text
                    start = text.lower().find(val.lower())
                    if start != -1:
                        end = start + len(val)
                        span = doc.char_span(start, end, label=label_name.upper(), alignment_mode="contract")
                        if span:
                            ents.append(span)
            
            # filter_spans handles overlapping entities (keeps the longest one)
            doc.ents = spacy.util.filter_spans(ents)
            doc_bin.add(doc)
            count += 1

    doc_bin.to_disk(output_file)
    print(f"Successfully processed {count} products into {output_file}")

if __name__ == "__main__":
    create_spacy_data("Dataset_Assignment.JSON", "train.spacy")

Successfully processed 51 products into train.spacy


In [ ]:
import spacy
from spacy.tokens import DocBin
import json
import os

def create_spacy_data(input_file, output_file):
    # Load a pretrained English pipeline with large word vectors
    nlp = spacy.load("en_core_web_lg")
    doc_bin = DocBin()
    
    count = 0
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): 
                continue
            item = json.loads(line)
            text = item['text']
            labels = item['labels']
            
            # Use the pretrained tokenizer
            doc = nlp.make_doc(text)
            ents = []
            
            for label_name, value in labels.items():
                # Split multiple values like "sage; dusty blue"
                sub_values = [v.strip() for v in str(value).split(';')]
                
                for val in sub_values:
                    if val.lower() in ["none", "unknown", "n/a"]:
                        continue
                    
                    # Find the exact position of the attribute in the text
                    start = text.lower().find(val.lower())
                    if start != -1:
                        end = start + len(val)
                        span = doc.char_span(
                            start, end, 
                            label=label_name.upper(), 
                            alignment_mode="contract"
                        )
                        if span:
                            ents.append(span)
            
            # filter_spans handles overlapping entities (keeps the longest one)
            doc.ents = spacy.util.filter_spans(ents)
            doc_bin.add(doc)
            count += 1

    doc_bin.to_disk(output_file)
    print(f"Successfully processed {count} products into {output_file}")

if __name__ == "__main__":
    create_spacy_data("Dataset_Assignment.JSON", "train.spacy")


# 1. Generate the configuration file
python -m spacy init config config.cfg --lang en --pipeline ner --optimize accuracy



# 2. Train the model (Using 20% of your data for validation/dev is standard, 
# but for a 50-item assignment, using the same file for both is okay to start)
python -m spacy train config.cfg --output ./output --paths.train train.spacy --paths.dev train.spacy


# 2. Train the model (using train.spacy as both train and dev for this assignment)
python -m spacy train config.cfg --output ./output --paths.train train.spacy --paths.dev train.spacy